In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../..")
from utils.scaling import fit_scalers, apply_scaler
from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_beus_articulatory_features

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
token_df = dataset_df_to_token_df(df)

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "cpu"

art_model = VideoAutoencoder(
    input_hw=(56, 112),
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
    input_channels=2,
    output_activation=None,
    stn_mode="separate",
    stn_transform="rigid"

).to(device)
ckpt = torch.load(
    "../encoder_training/ckpts/beus/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../encoder_training/ckpts/beus/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [16]:
import cv2
with open("../encoder_training/scalers/beus_video_stats.pkl", "rb") as f:
    _REC_STATS = pickle.load(f)

def recording_stats(path, n_sample=400):
    """
    Pixel mean and sd over the WHOLE recording, cached per file.
 
    Note cv2's CAP_PROP_POS_FRAMES seeking on compressed video snaps to the
    nearest keyframe on some codecs. That is harmless for statistics, but it is
    worth checking separately that _read_video_segment returns frames that really
    correspond to the requested times — a silent half-second offset would
    misalign every token against the audio.
    """
    if path in _REC_STATS:
        return _REC_STATS[path]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.unique(np.linspace(0, total - 1, min(n_sample, total)).astype(int))
    vals = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            vals.append(fr[:, :, 0].astype(np.float32))
    cap.release()
    if not vals:
        return None
    v = np.stack(vals)
    _REC_STATS[path] = (float(v.mean()), float(v.std()) + 1e-5)
    return _REC_STATS[path]

with open(f"../encoder_training/scalers/beus_ac.pkl", "rb") as f:
    scalers = pickle.load(f)

In [17]:
def get_vectors(art_f, ac_f, start, end):
    articulatory_features = get_beus_articulatory_features(
        art_f,
        start,
        end
    )

    if articulatory_features is not None:

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        art_out, art_z = art_model(
            articulatory_features,
            return_latent=True
        )
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = (
            torch.from_numpy(mfccs.T)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs)
    else:
        ac_out, ac_z = None, None

    return {
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [18]:
token_df

,start,end,phone,art_f,ac_f,subject
0,1.97,2.16,ɔ,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
1,1.47,1.53,p,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
2,2.39,2.50,ɛ,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
3,2.50,2.56,l,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
4,1.53,1.60,l,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
...,...,...,...,...,...,...
226483,0.55,0.64,t,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
226484,3.48,3.53,m,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
226485,0.93,1.09,t,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
226486,1.09,1.16,ɹ,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me


In [19]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    stats = recording_stats(art_f)
    articulatory_features = get_beus_articulatory_features(art_f, start, end, stats=stats)
    if articulatory_features is None: continue
    articulatory_features = torch.from_numpy(articulatory_features).unsqueeze(0).to(device)
    articulatory_features = articulatory_features.permute(0, 2, 1, 3, 4).contiguous()
    art_out, art_z = art_model(articulatory_features, return_latent=True)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = apply_scaler(mfccs, scalers[subject])
    mfccs = torch.from_numpy(mfccs).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs, return_latent=True)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)

100%|██████████| 226488/226488 [4:25:42<00:00, 14.21it/s]  


In [20]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (222964, 64) ac_flat: (222964, 64)


In [21]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

N = len(ac_raw)
k = 5

indices = np.arange(N)
np.random.shuffle(indices)
folds = np.array_split(indices, k)

In [22]:
art_dim = art_raw.shape[1]
ac_dim = ac_raw.shape[1]

In [23]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------
class AXDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# ----------------------------------------------------
# Training loop for one model
# ----------------------------------------------------
def train_model(train_loader, val_loader, x_dim, y_dim):
    model = MDN(x_dim=x_dim, y_dim=y_dim, n_components=5, hidden_dim=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = float('inf')
    patience = 10
    counter = 0

    for epoch in range(1, 2001):  # just large upper bound
        # ----- train -----
        model.train()
        train_loss = 0
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logp = model.log_prob(Xb, Yb)
            loss = -logp.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(Xb)
        train_loss /= len(train_loader.dataset)

        # ----- validation -----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                loss = -model.log_prob(Xb, Yb).mean()
                val_loss += loss.item() * len(Xb)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
            print("   ✓ New best model")
        else:
            counter += 1
            print(f"   No improvement ({counter} epochs)")

        if counter >= patience:
            print(f"Early stopping. Best val={best_val:.4f}")
            break

    model.load_state_dict(best_state)
    return model

In [24]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]


    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_ac = StandardScaler()
    sc_art = StandardScaler()

    sc_ac.fit(ac_raw[train_inner])
    sc_art.fit(art_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    ac_norm = sc_ac.transform(ac_raw)
    art_norm = sc_art.transform(art_raw)

    # Dataset for p(ART | AC)
    dataset = AXDataset(ac_norm, art_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=ac_raw.shape[1],
        y_dim=art_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            ac_fold,
            art_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    ART coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_art.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_art_given_ac_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_art_given_ac_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=39.9811, val=35.0030
   ✓ New best model
Epoch 2: train=32.3321, val=30.7308
   ✓ New best model
Epoch 3: train=29.3043, val=28.7785
   ✓ New best model
Epoch 4: train=27.2700, val=27.2321
   ✓ New best model
Epoch 5: train=25.7521, val=26.6687
   ✓ New best model
Epoch 6: train=24.6045, val=26.7122
   No improvement (1 epochs)
Epoch 7: train=23.6340, val=26.0307
   ✓ New best model
Epoch 8: train=22.8463, val=25.6891
   ✓ New best model
Epoch 9: train=22.1455, val=25.6164
   ✓ New best model
Epoch 10: train=21.5526, val=25.9193
   No improvement (1 epochs)
Epoch 11: train=21.0208, val=25.6685
   No improvement (2 epochs)
Epoch 12: train=20.5268, val=25.5498
   ✓ New best model
Epoch 13: train=20.0621, val=25.6083
   No improvement (1 epochs)
Epoch 14: train=19.6586, val=25.4935
   ✓ New best model
Epoch 15: train=19.2430, val=25.2220
   ✓ New best model
Epoch 16: train=18.9969, val=25.3349
   No improvement (1 epochs)
Epoch 17: train=18.6305, val=25.8972
   N

In [25]:
import pickle
with open("entropies/beus_art_given_ac.pickle", "wb") as f:
    pickle.dump(H_art_given_ac_mdn, f)

In [26]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]

    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_art = StandardScaler()
    sc_ac = StandardScaler()

    sc_art.fit(art_raw[train_inner])
    sc_ac.fit(ac_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    art_norm = sc_art.transform(art_raw)
    ac_norm = sc_ac.transform(ac_raw)

    # Dataset for p(AC | ART)
    dataset = AXDataset(art_norm, ac_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=art_raw.shape[1],
        y_dim=ac_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            art_fold,
            ac_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    AC coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_ac.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_ac_given_art_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_ac_given_art_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=87.0748, val=86.6331
   ✓ New best model
Epoch 2: train=86.4710, val=86.3294
   ✓ New best model
Epoch 3: train=86.2641, val=86.2248
   ✓ New best model
Epoch 4: train=86.1228, val=86.1241
   ✓ New best model
Epoch 5: train=86.0218, val=86.0159
   ✓ New best model
Epoch 6: train=85.9499, val=86.0090
   ✓ New best model
Epoch 7: train=85.8908, val=85.9607
   ✓ New best model
Epoch 8: train=85.8439, val=85.9312
   ✓ New best model
Epoch 9: train=85.8073, val=85.9028
   ✓ New best model
Epoch 10: train=85.7718, val=85.9080
   No improvement (1 epochs)
Epoch 11: train=85.7499, val=85.8821
   ✓ New best model
Epoch 12: train=85.7232, val=85.8611
   ✓ New best model
Epoch 13: train=85.7015, val=85.8567
   ✓ New best model
Epoch 14: train=85.6869, val=85.8691
   No improvement (1 epochs)
Epoch 15: train=85.6692, val=85.8662
   No improvement (2 epochs)
Epoch 16: train=85.6601, val=85.8447
   ✓ New best model
Epoch 17: train=85.6470, val=85.8594
   No improvement (1 e

In [27]:
with open("entropies/beus_ac_given_art.pickle", "wb") as f:
    pickle.dump(H_ac_given_art_mdn, f)